# 04 - Factor Evaluation

Every factor is judged by the **same** harness, so results are comparable. For
each factor: multi-horizon IC (1d / 5d / 21d), the Newey-West t-stat
(overlap-aware - the daily ICs against a 21-day forward return overlap, so the
naive t-stat is inflated), a decile long-short return, and then a
Benjamini-Hochberg correction and deflated Sharpe **across the whole zoo**.

The headline question this answers: which factors survive multiple testing,
not which one looked best in isolation. `factor_long_short` now applies a 1-day execution lag by default, so the LS numbers
here and in the zoo are the *tradeable* ones (signal at T, traded at T+1).

**Prerequisite:** `make data`.

In [1]:
from qer.data.loader import DataLoader
from qer.factors import all_factors, get_factor
from qer.diagnostics.factor_ic import compute_factor_ic, summarize_ic
import pandas as pd, numpy as np
H = 21  # primary forward horizon (trading days); NW lag = H-1 captures the overlap

loader = DataLoader()
close = loader.close
start = close.index.max() - pd.DateOffset(years=5)
dates = close.index[(close.index >= start) & (close.index <= close.index[-22])]
dates = dates[dates >= close.index[273]]
len(dates)

1235

## Multi-horizon IC for one factor

`compute_factor_ic` returns `{horizon: IC series}`. Reading the decay across
horizons tells you the signal's natural holding period.

In [2]:
ic = compute_factor_ic(loader, get_factor("momentum_12_1"), horizons=(1,5,21), dates=dates)
pd.Series({h: ic[h].mean() for h in (1,5,21)}, name="mean_IC")

1     0.016348
5     0.009719
21    0.005209
Name: mean_IC, dtype: float64

## Naive vs Newey-West t-stat

On daily ICs against a 21-day forward return, neighbouring observations share 20
of 21 days. The naive `mean/std*sqrt(N)` t-stat ignores that; the Newey-West
t-stat (lag = 20) corrects it and is the one to report.

In [3]:
s = summarize_ic(ic[H], newey_west_lags=H-1)
{k: round(v,4) for k,v in s.items() if k in ("mean_ic","ic_ir_annualized","t_stat","t_stat_nw","hit_rate","n")}

{'n': 1235,
 'mean_ic': np.float64(0.0052),
 'ic_ir_annualized': np.float64(0.4318),
 't_stat': np.float64(0.9558),
 'hit_rate': np.float64(0.5506),
 't_stat_nw': np.float64(0.2707)}

## The whole zoo in one table

`run_factor_zoo` evaluates every available factor and applies the cross-factor
corrections. The same logic is callable here so the notebook can show the table
inline.

In [4]:
from qer.diagnostics.factor_zoo import build_factor_zoo_table
table = build_factor_zoo_table(loader, years=5)
table

/home/alex/Desktop/quant-equity-research/src/qer/data/loader.py:293: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change()
/home/alex/Desktop/quant-equity-research/src/qer/data/loader.py:293: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change()
/home/alex/Desktop/quant-equity-research/.venv/lib/python3.12/site-packages/pandas/core/window/rolling.py:611: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)
/home/alex/Desktop/quant-equity-research/.venv/lib/python3.12/site-packages/pandas/core/window/rolling.py:

,direction,mean_ic_21d,ic_ir,t_naive,t_nw,hit_rate,ls_sharpe,n,p_value,survives_bh,deflated_sharpe_ratio
factor,,,,,,,,,,,
amihud_illiquidity,1,-0.015898,-2.031548,-4.497387,-1.250921,0.583806,-0.104127,1235,0.210963,False,0.018213
momentum_12_1,1,0.005209,0.431767,0.955835,0.270722,0.550607,0.190920,1235,0.786605,False,0.551153
quality_gp,1,-0.005821,-0.880467,-1.949155,-0.528037,0.539271,-0.100602,1235,0.597473,False,0.019415
reversal_1m,-1,0.009979,0.891860,1.974377,0.596963,0.514170,0.011795,1235,0.550532,False,0.110745
size,-1,-0.004525,-0.638779,-1.414112,-0.382094,0.548988,0.041963,1235,0.702391,False,0.160008
idio_skew_60d,-1,-0.005102,-0.944393,-2.090673,-0.608114,0.510121,-0.115132,1235,0.543112,False,0.014864
value_btm,1,0.015528,1.633441,3.616069,0.991099,0.532794,0.164459,1235,0.321637,False,0.472270
volatility_60d,-1,-0.009655,-0.628296,-1.390906,-0.384391,0.483401,0.042581,1235,0.700689,False,0.161150


## Turnover and significance for one factor

The zoo table already gives each factor's long-short Sharpe and its deflated-Sharpe
p-value. What it does **not** show is turnover - the cost driver. Here we add turnover
for a factor of interest and read its significance straight off the zoo table, rather
than recomputing a bare (and un-annualised) Sharpe that has no inference attached.

In [5]:
from qer.diagnostics.portfolios import long_short_turnover
from qer.factors.base import compute_factor_panel
name = "momentum_12_1"
fac = get_factor(name)
# Two-leg turnover (the cost-relevant one); the zoo holds Sharpe + significance.
turn = long_short_turnover(compute_factor_panel(loader, fac, dates=dates, oriented=True))
row = table.loc[name]
print(f"{name}: mean two-leg turnover {turn.mean():.3f}/rebalance "
      f"| LS Sharpe {row['ls_sharpe']:+.3f} "
      f"| deflated-Sharpe ratio {row['deflated_sharpe_ratio']:.3f} "
      f"| survives BH: {bool(row['survives_bh'])}")

momentum_12_1: mean two-leg turnover 0.117/rebalance | LS Sharpe +0.191 | deflated-Sharpe ratio 0.551 | survives BH: False


## Takeaways

Read the zoo table by the `survives_bh` and `deflated_sharpe_p` columns, not by
raw mean IC. A factor with a high IC that fails the correction is not a finding.
The `t_nw` column is the honest significance; `t_naive` is the inflated one we
do **not** report.